# Testing RMSE untuk Top 5 Arms Semua Pengguna

**Tujuan:** Membandingkan MAPE vs RMSE untuk evaluasi akurasi nutrisi arms

## 1. Import Libraries & Load Data

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# Load data dari semua pengguna
file_path = "KUMPULAN TOP 5 SEMUA PENGGUNA.xlsx"

# Baca semua sheets
xl = pd.ExcelFile(file_path)
print(f"Total sheets: {len(xl.sheet_names)}")
print(f"Sheet names: {xl.sheet_names}")

# Load data dari setiap pengguna
data_pengguna = {}
for sheet_name in xl.sheet_names:
    data_pengguna[sheet_name] = pd.read_excel(file_path, sheet_name=sheet_name)
    print(f"✓ {sheet_name}: {len(data_pengguna[sheet_name])} arms")

Total sheets: 5
Sheet names: ['Pengguna 1', 'Pengguna 2', 'Pengguna 3', 'Pengguna 4', 'Pengguna 5']
✓ Pengguna 1: 5 arms
✓ Pengguna 2: 5 arms
✓ Pengguna 3: 5 arms
✓ Pengguna 4: 5 arms
✓ Pengguna 5: 5 arms


## 2. Definisi Target Nutrisi per Pengguna

**PENTING:** Setiap pengguna punya TDEE berbeda, jadi target nutrisi juga berbeda!

In [8]:
# Fungsi hitung kebutuhan nutrisi dari TDEE
def hitung_kebutuhan_nutrisi(tdee):
    """
    Hitung kebutuhan nutrisi berdasarkan TDEE
    Sesuai dengan DASH diet guidelines
    """
    return {
        "kalori": tdee,
        "karbohidrat": (tdee * 0.55) / 4,    # 55% kalori, 1g = 4 kkal
        "protein": (tdee * 0.18) / 4,         # 18% kalori, 1g = 4 kkal
        "lemak": (tdee * 0.27) / 9,           # 27% kalori, 1g = 9 kkal
        "lemak_jenuh": (tdee * 0.06) / 9,     # 6% kalori, 1g = 9 kkal
        "kolesterol": 150,                    # mg 
        "natrium": 2300,                      # mg 
        "kalium": 4700,                       # mg 
        "serat": 30                           # g 
    }

# Target TDEE per pengguna 
TDEE_PENGGUNA = {
    "Pengguna 1": 1519,  
    "Pengguna 2": 2000,  
    "Pengguna 3": 1645,  
    "Pengguna 4": 2591,  
    "Pengguna 5": 3018,  
}

# Hitung target nutrisi untuk setiap pengguna
target_nutrisi = {}
for pengguna, tdee in TDEE_PENGGUNA.items():
    target_nutrisi[pengguna] = hitung_kebutuhan_nutrisi(tdee)
    print(f"\n{pengguna} (TDEE: {tdee} kkal):")
    print(f"  Karbohidrat: {target_nutrisi[pengguna]['karbohidrat']:.1f} g")
    print(f"  Protein: {target_nutrisi[pengguna]['protein']:.1f} g")
    print(f"  Lemak: {target_nutrisi[pengguna]['lemak']:.1f} g")
    print(f"  Lemak Jenuh: {target_nutrisi[pengguna]['lemak_jenuh']:.1f} g")
    print(f"  Kolesterol: {target_nutrisi[pengguna]['kolesterol']:.1f} g")
    print(f"  Natrium: {target_nutrisi[pengguna]['natrium']:.1f} g")
    print(f"  Kalium: {target_nutrisi[pengguna]['kalium']:.1f} g")
    print(f"  Serat: {target_nutrisi[pengguna]['serat']:.1f} g")


Pengguna 1 (TDEE: 1519 kkal):
  Karbohidrat: 208.9 g
  Protein: 68.4 g
  Lemak: 45.6 g
  Lemak Jenuh: 10.1 g
  Kolesterol: 150.0 g
  Natrium: 2300.0 g
  Kalium: 4700.0 g
  Serat: 30.0 g

Pengguna 2 (TDEE: 2000 kkal):
  Karbohidrat: 275.0 g
  Protein: 90.0 g
  Lemak: 60.0 g
  Lemak Jenuh: 13.3 g
  Kolesterol: 150.0 g
  Natrium: 2300.0 g
  Kalium: 4700.0 g
  Serat: 30.0 g

Pengguna 3 (TDEE: 1645 kkal):
  Karbohidrat: 226.2 g
  Protein: 74.0 g
  Lemak: 49.4 g
  Lemak Jenuh: 11.0 g
  Kolesterol: 150.0 g
  Natrium: 2300.0 g
  Kalium: 4700.0 g
  Serat: 30.0 g

Pengguna 4 (TDEE: 2591 kkal):
  Karbohidrat: 356.3 g
  Protein: 116.6 g
  Lemak: 77.7 g
  Lemak Jenuh: 17.3 g
  Kolesterol: 150.0 g
  Natrium: 2300.0 g
  Kalium: 4700.0 g
  Serat: 30.0 g

Pengguna 5 (TDEE: 3018 kkal):
  Karbohidrat: 415.0 g
  Protein: 135.8 g
  Lemak: 90.5 g
  Lemak Jenuh: 20.1 g
  Kolesterol: 150.0 g
  Natrium: 2300.0 g
  Kalium: 4700.0 g
  Serat: 30.0 g


## 3. Fungsi MAPE dengan Toleransi ±10%

**Sesuai dengan implementasi di `implementasi_cmab.py`**

In [9]:
def hitung_mape(arm_row, kebutuhan_nutrisi, toleransi=0.10):
    """
    Hitung MAPE dengan toleransi range ±10%
    
    NUTRISI BIASA (harus dipenuhi):
    - Range: 90-110% dari target
    - Error jika < 90% ATAU > 110%
    
    NUTRISI DIBATASI (lemak jenuh, kolesterol, natrium):
    - Range: 0-110% dari target (semakin rendah semakin bagus)
    - Error HANYA jika > 110%
    """
    
    # Daftar nutrisi yang harus dibatasi (semakin rendah semakin bagus)
    NUTRISI_DIBATASI = {3, 4, 5}  # lemak_jenuh, kolesterol, natrium
    
    target = [
        kebutuhan_nutrisi["karbohidrat"],     # 0
        kebutuhan_nutrisi["protein"],          # 1
        kebutuhan_nutrisi["lemak"],            # 2
        kebutuhan_nutrisi["lemak_jenuh"],      # 3 ← DIBATASI
        kebutuhan_nutrisi["kolesterol"],       # 4 ← DIBATASI
        kebutuhan_nutrisi["natrium"],          # 5 ← DIBATASI
        kebutuhan_nutrisi["kalium"],           # 6
        kebutuhan_nutrisi["serat"]             # 7
    ]
    
    aktual = [
        arm_row["total_karbohidrat"],          # 0
        arm_row["total_protein"],              # 1
        arm_row["total_lemak"],                # 2
        arm_row["total_lemak_jenuh"],          # 3 ← DIBATASI
        arm_row["total_kolesterol"],           # 4 ← DIBATASI
        arm_row["total_natrium"],              # 5 ← DIBATASI
        arm_row["total_kalium"],               # 6
        arm_row["total_serat"]                 # 7
    ]
    
    percentage_errors = []
    
    for idx, (a, t) in enumerate(zip(aktual, target)):
        if t <= 0:
            continue
        
        is_dibatasi = idx in NUTRISI_DIBATASI
        
        if is_dibatasi:
            # NUTRISI DIBATASI
            batas_atas = t * (1 + toleransi)
            if a <= batas_atas:
                error = 0
            else:
                error = abs(a - batas_atas) / batas_atas * 100
        else:
            # NUTRISI BIASA
            batas_bawah = t * (1 - toleransi)
            batas_atas = t * (1 + toleransi)
            
            if batas_bawah <= a <= batas_atas:
                error = 0
            elif a < batas_bawah:
                error = abs(a - batas_bawah) / batas_bawah * 100
            else:
                error = abs(a - batas_atas) / batas_atas * 100
        
        percentage_errors.append(error)
    
    mape = sum(percentage_errors) / len(percentage_errors) if percentage_errors else 0
    return mape

## 4. Fungsi RMSE (Root Mean Square Error)

**Formula:**
$$
RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
$$

In [10]:
def hitung_rmse(arm_row, kebutuhan_nutrisi, toleransi=0.10):
    """
    Hitung MAPE dengan toleransi range ±10%
    
    NUTRISI BIASA (harus dipenuhi):
    - Range: 90-110% dari target
    - Error jika < 90% ATAU > 110%
    
    NUTRISI DIBATASI (lemak jenuh, kolesterol, natrium):
    - Range: 0-110% dari target (semakin rendah semakin bagus)
    - Error HANYA jika > 110%
    """
    
    NUTRISI_DIBATASI = {3, 4, 5}
    
    # Target dalam gram (konversi mg → g)
    target = [
        kebutuhan_nutrisi["karbohidrat"],
        kebutuhan_nutrisi["protein"],
        kebutuhan_nutrisi["lemak"],
        kebutuhan_nutrisi["lemak_jenuh"],
        kebutuhan_nutrisi["kolesterol"] / 1000,
        kebutuhan_nutrisi["natrium"] / 1000,
        kebutuhan_nutrisi["kalium"] / 1000,
        kebutuhan_nutrisi["serat"]
    ]

     # Aktual dalam gram
    aktual = [
        arm_row["total_karbohidrat"],
        arm_row["total_protein"],
        arm_row["total_lemak"],
        arm_row["total_lemak_jenuh"],
        arm_row["total_kolesterol"] / 1000,
        arm_row["total_natrium"] / 1000,
        arm_row["total_kalium"]/ 1000,
        arm_row["total_serat"]
    ]
    
    squared_errors = []
    
    for idx, (a, t) in enumerate(zip(aktual, target)):
        if t <= 0:
            continue
        
        is_dibatasi = idx in NUTRISI_DIBATASI
        
        if is_dibatasi:
            # ============================================
            # NUTRISI DIBATASI (lemak jenuh, kolesterol, natrium)
            # Error hanya jika > 110%
            # ============================================
            batas_atas = t * (1 + toleransi)  # 110%
            
            if a <= batas_atas:
                # ≤ 110% → BAGUS! Error = 0
                error = 0
            else:
                # > 110% → Hitung error dari batas atas (dalam gram)
                error = a - batas_atas
        
        else:
            # ============================================
            # NUTRISI BIASA (harus dipenuhi)
            # Error jika < 90% atau > 110%
            # ============================================
            batas_bawah = t * (1 - toleransi)  # 90%
            batas_atas = t * (1 + toleransi)   # 110%
            
            if batas_bawah <= a <= batas_atas:
                # DALAM RANGE → Error = 0
                error = 0
            elif a < batas_bawah:
                # DI BAWAH RANGE → Error dari batas bawah
                error = batas_bawah - a
            else:  # a > batas_atas
                # DI ATAS RANGE → Error dari batas atas
                error = a - batas_atas
        
        # Kuadratkan error
        squared_errors.append(error ** 2)
    
    # RMSE = sqrt(rata-rata error²)
    mse = sum(squared_errors) / len(squared_errors) if squared_errors else 0
    rmse = np.sqrt(mse)
    
    return rmse

## 5. Hitung MAPE & RMSE untuk Semua Pengguna

In [11]:
# Hitung MAPE, RMSE, dan NRMSE untuk setiap arm
results = []

for pengguna, df in data_pengguna.items():
    target = target_nutrisi[pengguna]
    
    for idx, row in df.iterrows():
        mape = hitung_mape(row, target)
        rmse = hitung_rmse(row, target) 
        
        results.append({
            "pengguna": pengguna,
            "arm_id": row["arm_id"],
            "mape": mape,
            "rmse": rmse
        })

# Convert to DataFrame
df_results = pd.DataFrame(results)

print("\nHASIL PERHITUNGAN:")
display(df_results)
print(f"\nTotal data: {len(df_results)} arms (5 pengguna × 5 arms = 25)")


HASIL PERHITUNGAN:


,pengguna,arm_id,mape,rmse
0,Pengguna 1,3488,2.924193,1.844004
1,Pengguna 1,4550,3.327864,3.860398
2,Pengguna 1,8813,7.439192,4.215500
3,Pengguna 1,2807,7.448965,4.550999
4,Pengguna 1,4969,4.067263,20.653744
5,Pengguna 2,7949,5.365136,4.938826
6,Pengguna 2,796,5.770039,4.528808
7,Pengguna 2,5298,1.009786,1.180869
8,Pengguna 2,6773,7.040225,3.152488
9,Pengguna 2,641,3.998159,7.511633



Total data: 25 arms (5 pengguna × 5 arms = 25)
